# NB0 – Projektübersicht & Projektplan
### CAS Information Engineering – Scripting Project

**Gruppe:** SC26_Gruppe_2 (Test) 
**Mitglieder:** Patrik Neunteufel · Senthuran Elankeswaran · Cyril Saladin  
**Datum:** März–Mai 2026

---

Dieses Notebook ist der **Einstiegspunkt** für das gesamte Projekt. Es ersetzt den separaten Projektplan (Projektplan_Grid_Arbitrage.docx) und beschreibt Motivation, Zielsetzung, Methodik, Datenquellen, Qualitätssicherung sowie den Projektplan mit Aufgabenverteilung.


---
## 1. Motivation

### Extrinsische Motivation — Leistungsnachweis

Dieses Projekt ist der Leistungsnachweis für den Kurs *Scripting* im Rahmen des CAS Information Engineering an der ZHAW School of Engineering. Die Aufgabe erfordert:

- Eigenständige **Datenbeschaffung** aus offenen Quellen (APIs, Open Data)
- **Datenbereinigung und -analyse** mit Python/Pandas in Jupyter Notebooks
- **Visualisierung** der Ergebnisse mit mindestens zwei Diagrammtypen (inkl. Heatmap)
- Einen **Business Case** als viertes Notebook

### Intrinsische integrale Motivation — Beruflicher Kontext

Für Teile der Gruppe deckt sich die Aufgabenstellung direkt mit dem beruflichen Betätigungsfeld: Leistungselektronik, Batteriespeicher, Energiemärkte und die wirtschaftliche Bewertung dezentraler Flexibilitätsressourcen. Das Projekt erlaubt es, akademische Anforderungen mit echtem analytischen Erkenntnisinteresse zu verbinden.

Konkret: Das Ergebnis dieses Projekts kann als Entscheidungsgrundlage für **reale Investitionsentscheide** in Batteriespeicher-Infrastruktur dienen, oder auch der Produktentwicklung in diesem Segment — insbesondere da der Schweizer Flexibilitätsmarkt (Swissgrid) in den nächsten Jahren geöffnet wird.



---
## 2. Business Case

Grid-Arbitrage mit Batteriespeichern ermöglicht die wirtschaftliche Optimierung von Strombezug und Einspeisung durch die Nutzung zeitlicher Preisunterschiede. In der Schweiz führt der steigende Anteil erneuerbarer Energien zu höherer Volatilität bei Strompreisen und Netzlast — wodurch sich systematische Potenziale für speicherbasierte Arbitrage ergeben.

**Forschungsfrage:**
> Unter welchen Bedingungen ist Grid-Arbitrage mit Batteriespeichern in der Schweiz wirtschaftlich sinnvoll — und welchen systemischen Beitrag zur Netzentlastung können dezentrale Speicher dabei leisten?

> **Hinweis:** Der Pflichtteil (NB01–NB04) beantwortet den ersten Teil quantitativ: rentabel oder nicht, und warum. Die vollständige Antwort — unter welchen Markt-, Technologie- und Regulierungsbedingungen sich das Bild dreht — liefert [NB05 Business Strategy](05_Business_Strategy.ipynb) als Kür.

### Stakeholder

| Stakeholder | Primäres Interesse |
|-------------|-------------------|
| Netzbetreiber (Swissgrid) | Netzstabilität, geringerer Ausbaubedarf |
| Privathaushalte | Kostenersparnis beim Strombezug |
| Gewerbe / Industrie | Lastspitzenreduktion und Arbitrage-Erlöse |
| Energieversorger / Investoren | Portfolio-Optimierung |
| Politik / BFE | Entscheidungsgrundlage für Energiestrategie 2050 |


---
## 3. Datenquellen

Das Projekt verwendet **3 Hauptdatensätze** gemäss Projektanforderung. Weitere Datensätze für die optionalen Kür-Notebooks sind separat ausgewiesen.

### Hauptdatensätze (Pflicht)

| # | Quelle | Datensatz | Format | Zeitraum | Notebook |
|---|--------|-----------|--------|----------|----------|
| DS1 | **ENTSO-E** | Day-Ahead Spot-Preise CH (`10YCH-SWISSGRIDZ`) | API (entsoe-py) | 2023–heute | NB01 |
| DS2 | **ENTSO-E** | Netzlast CH — stündliche Systemlast [MW] | API (query_load) | 2023–heute | NB01 |

> DS1 und DS2 stammen von derselben Quelle (ENTSO-E Transparency Platform) — für den Moodle-Eintrag reicht eine URL für beide.

### Weitere Datensätze (Kür — werden im ersten NB geladen, das sie benötigt)

| Quelle | Datensatz | Format | Geladen in |
|--------|-----------|--------|------------|
| **ENTSO-E** | Grenzflüsse CH↔DE/AT/IT/FR (Netto-Export) | API (crossborder_flows) | NB07 Sektion 1 |
| **BFE** | Elektrizitätsproduktionsanlagen (322k Anlagen) | GeoPackage EPSG:2056 | NB06 Sektion 1.1 |
| **BFS** | STATPOP — Kantonsbevölkerung | PXWeb API | NB06 Sektion 1.2 |
| **swisstopo** | swissBOUNDARIES3D — Kantonsgrenzen | Shapefile/GeoJSON | NB06 Sektion 1.3 |

: Datenzeitraum

Der Datenzeitraum wird zentral in `config.json → daten.start_year / end_year` gesteuert.  
Mit `end_year: "heute"` lädt NB01 automatisch alle verfügbaren Jahre bis zum aktuellen Zeitpunkt.

**Warum ab 2023:** Vor 2023 waren die CH Day-Ahead-Preise durch den Energiekrisen-Peak 2021/2022 stark verzerrt -> kein repräsentativer Referenzzeitraum für eine strukturelle Analyse.

**Warum dynamisch bis heute:** Arbitrage-Potenzial und CAPEX entwickeln sich laufend. Das Skript kann jährlich neu ausgeführt werden um Trendaussagen zu aktualisieren.


---
## 4. Methodisches Vorgehen

### 4.1 Datengrundlage

| Datensatz | Quelle | Zeitraum | Zweck |
|-----------|--------|----------|-------|
| Day-Ahead Spot-Preise CH | ENTSO-E API | 2023–heute | Primäre Entscheidungsgrösse Dispatch |
| Systemlast CH | ENTSO-E API | 2023–heute | Netzentlastungsmodell |
| Produktionsanlagen | BFE GeoPackage | aktuell | Räumliche Analyse (Kür) |
| Grenzflüsse CH | ENTSO-E API | 2023–heute | Empirische Validierung (Kür) |

### 4.2 Analysevorgehen

Das Projekt folgt einer strukturierten, reproduzierbaren Datenpipeline:

```
Externe Quellen
    │
    ▼
NB01 Datenbeschaffung      API / CSV / GPKG  →  data/raw/
    │                      └─ schreibt → transfer.json [datenzeitraum]
    ▼
NB02 Bereinigung &         UTC-Normierung, Lücken, Ausreisser
     Feature Engineering   +hour, +month, +season  →  data/processed/
     Analyse & Simulation  Batterie-Dispatch, ROI/CAPEX
                           →  data/intermediate/<szenario>/
    │                      └─ schreibt → transfer.json [simulation]
    ▼
NB03 Visualisierung        Charts 1–8, Animationen A/B/C  →  output/charts/
    │
    ▼
NB04 Business Case         liest ← transfer.json [datenzeitraum, simulation]
── Kür ────────────────────────────────────────────────────────────────────────
NB05 Business Strategy     liest ← transfer.json [alle Sektionen]
NB06 Räumliche Analyse     Geodaten, Zonenbilanzen, BVI
NB07 Cross-Border          Grenzflüsse, Import/Export
NB08 Marktdynamik          Spread-Trend, CAPEX-Lernkurve
NB09 Revenue Stacking      liest ← transfer.json
NB10 Dispatch-Optimierung  schreibt → transfer.json [dispatch_optimierung]
NB13 Eigenverbrauch        schreibt → transfer.json [eigenverbrauch]
NB14 Produktsteckbrief     liest ← / schreibt → transfer.json [produkt]
NB15 Komb. Simulation      schreibt → transfer.json [hybrid_simulation]
```

### 4.3 Dispatch-Modell & Bewertungskriterien

Tagesbasiertes Schwellenwertmodell:
- **Laden** wenn Preis < p25 des Tages und SoC < 95%
- **Einspeisen** wenn Preis > p75 des Tages und SoC > 5%
- **Idle** sonst

Bewertungskriterien: Jahreserlös [EUR], ROI [%], Amortisationsdauer [Jahre], Netzentlastung [MW].

### 4.4 Zentrales Modell: Annahmen und Limitationen

**Explizite Annahmen:**

| Parameter | Wert | Begründung |
|-----------|------|------------|
| Round-Trip-Wirkungsgrad | 92% | Konservativer Li-Ion-Wert (Literatur: 90–95%) |
| CAPEX Privat | 400 EUR/kWh | Marktüblicher CH Systempreis 2024 (inkl. Wechselrichter) |
| OPEX | 1.5% CAPEX/Jahr | Branchenübliche Schätzung Wartung/Versicherung |
| Simulationshorizont | 12 Jahre | Typische Betriebsdauer Li-Ion (80% Kapazitätserhalt) |
| Technologie | Li-Ion | Dominanter Typ im CH Heimspeicher- und Gewerbemarkt |

**Modell-Limitationen:**

- Das Dispatch-Modell entscheidet täglich auf Basis von Tages-Quantilen — dies approximiert einen DA-optimalen Dispatcher (ENTSO-E DA-Preise sind vor Tagesbeginn bekannt), unterschätzt aber mögliche Intraday-Gewinne
- Keine Modellierung von Degradation über die Betriebsdauer
- Netzentlastungsszenarien basieren auf geschätzten Rollout-Zahlen, keine offiziellen BFE-Ziele
- Keine Berücksichtigung von Steuern, Förderbeiträgen oder Netztarifen

### 4.5 Ergebnisoffenheit

Ein negatives Ergebnis — Grid-Arbitrage ist bei aktuellen CH Day-Ahead-Preisen nicht rentabel — ist kein Projektfehler, sondern ein valider Befund. Die Kür-Notebooks (NB08–NB10) untersuchen systematisch, unter welchen Bedingungen sich das Bild dreht.


---
## 4.6 Konfigurationsdateien: config.json und transfer.json

Das Projekt trennt **statische Inputs** (Schalter, Parameter) von **berechneten Outputs** (Simulationsergebnisse) in zwei separate JSON-Dateien.

### config.json — Single Source of Truth für alle Parameter

Wird vom **User** gepflegt. Notebooks lesen nur, schreiben nie.

| Sektion | Schlüssel (Beispiel) | Beschreibung |
|---------|----------------------|--------------|
| `mode` | `"data"` | `"data"` = echte Daten, `"sim"` = simulierte Daten |
| `eur_chf` | `0.97` | Fixkurs EUR→CHF (März 2026). Alle Berechnungen in EUR. |
| `daten` | `start_year`, `end_year` | Datenzeitraum; `"heute"` = aktuelles Jahr automatisch |
| `force_reload` | `spot_prices`, `netzlast`, … | `true` = Datei neu laden auch wenn vorhanden |
| `pflicht.simulation` | `charge_quantile`, `efficiency_roundtrip`, … | Dispatch-Modell-Parameter |
| `pflicht.wirtschaftlichkeit` | `capex_eur_kwh`, `opex_rate`, `lifetime_j`, … | Wirtschaftlichkeitsrechnung |
| `szenarien` | `gleichzeitigkeit_aktiv`, `optionen.realistisch.rate`, … | VPP-Szenarien |
| `kuer` | `markt.capex_ziel_privat_eur_kwh`, … | Parameter ausschliesslich für Kür-NBs |
| `kuer_aktiv` | `NB06: true`, `NB15: true`, … | Einreichungssteuerung Kür-Notebooks |
| `api_keys` | `entsoe` | API-Schlüssel — nie im Notebook-Code hardcoden |

```python
# Laden in jedem Notebook (erste Code-Zelle nach Imports):
import json as _json
with open('config.json') as _f:
    CFG = _json.load(_f)
MODE    = CFG['mode']          # Alias — nur lesend, nie direkt setzen
EUR_CHF = CFG.get('eur_chf', 0.97)
```

### transfer.json — Berechnete Zwischenergebnisse zwischen Notebooks

Wird von **Notebooks** geschrieben. Jedes Notebook überschreibt nur seine eigene Sektion; alle anderen Sektionen bleiben unverändert.

| Sektion | Geschrieben von | Gelesen von | Inhalt |
|---------|-----------------|-------------|--------|
| `datenzeitraum` | NB01 | NB02–NB15 | `start_date`, `end_date`, `n_years` aus echten Preisdaten |
| `simulation` | NB02 | NB04, NB05, NB09, NB14, NB15 | Spread-Kennzahlen, ROI/Payback je Segment |
| `dispatch_optimierung` | NB10 | NB05 | DA-optimal vs. reaktiv je Segment |
| `eigenverbrauch` | NB13 | NB14 | ROI, Jahresersparnis Eigenverbrauchsmodell |
| `hybrid_simulation` | NB15 | NB14 | ROI aller 4 Dispatch-Modelle je Segment |
| `produkt` | NB14 | NB05 | Steckbrief-Kennzahlen des konkreten Produkts |

```python
# Sicher laden (robust gegen leere Datei zu Projektbeginn):
import os as _os, json as _json
_tf_path = 'transfer.json'
_tf = _json.loads(open(_tf_path).read() or '{}') \
      if _os.path.exists(_tf_path) and _os.path.getsize(_tf_path) > 0 else {}
```

```python
# Schreiben (nur eigene Sektion — andere bleiben erhalten):
_tf['simulation'] = { ... }                           # nur diese Sektion ändern
with open(_tf_path, 'w') as _f:
    _json.dump(_tf, _f, indent=2, ensure_ascii=False) # Rest bleibt unverändert
```

> **Ausführungsreihenfolge für vollständiges transfer.json:**  
> NB01 → NB02 → NB10 → NB13 → NB15 → NB14  
> Ein einzelnes NB kann jederzeit neu ausgeführt werden — es überschreibt nur seine eigene Sektion.


---
## 5. Notebook-Map

### Pflicht

| NB | Datei | Status | Verantwortlich | Link |
|----|-------|--------|----------------|------|
| NB00 | `00_Project_Overview.ipynb` | ✅ abgeschlossen | Alle | ← dieses Notebook |
| NB00a | `00a_Glossar.ipynb` | ✅ abgeschlossen | Alle | [öffnen](00a_Glossar.ipynb) |
| NB00b | `00b_Konventionen.ipynb` | ✅ abgeschlossen | Alle | [öffnen](00b_Konventionen.ipynb) |
| NB01 | `01_Daten_Laden.ipynb` | 🔄 in Arbeit | PN | [öffnen](01_Daten_Laden.ipynb) |
| NB02 | `02_Daten_Analyse.ipynb` | 🔄 in Arbeit | SE | [öffnen](02_Daten_Analyse.ipynb) |
| NB03 | `03_Visualisierungen.ipynb` | 🔄 in Arbeit | CS | [öffnen](03_Visualisierungen.ipynb) |
| NB04 | `04_Business_Case.ipynb` | 🔄 in Arbeit | Alle | [öffnen](04_Business_Case.ipynb) |

### Kür

> **🔒 Eingefroren per 30.03.2026** — Keine neuen Notebooks ab diesem Datum.
> Neue Ideen werden ausschliesslich als *Potenzielle Erweiterungen* am Ende dieser Sektion notiert.
> Am **30.04.2026** wird entschieden, welche Kür-Notebooks mit dem Projekt eingereicht werden.

| NB | Datei | Status | Einreichung? | Link |
|----|-------|--------|--------------|------|
| NB05 | `05_Business_Strategy.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](05_Business_Strategy.ipynb) |
| NB06 | `06_Raeumliche_Analyse.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](06_Raeumliche_Analyse.ipynb) |
| NB07 | `07_Cross_Border.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](07_Cross_Border.ipynb) |
| NB08 | `08_Marktdynamik.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](08_Marktdynamik.ipynb) |
| NB08a | `08a_Animationen.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](08a_Animationen.ipynb) |
| NB09 | `09_Revenue_Stacking.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](09_Revenue_Stacking.ipynb) |
| NB10 | `10_Dispatch_Optimierung.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](10_Dispatch_Optimierung.ipynb) |
| NB11 | `11_Technologievergleich.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](11_Technologievergleich.ipynb) |
| NB12 | `12_Alternative_Speicher.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](12_Alternative_Speicher.ipynb) |
| NB13 | `13_Eigenverbrauch.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](13_Eigenverbrauch.ipynb) |
| NB14 | `14_Produkt_Steckbrief.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](14_Produkt_Steckbrief.ipynb) |
| NB15 | `15_Kombinierte_Simulation.ipynb` | 🔄 in Arbeit | 🕐 offen (30.04.) | [öffnen](15_Kombinierte_Simulation.ipynb) |

### Potenzielle Erweiterungen *(nicht Teil der Abgabe — nur Ideen)*

> Neue Ideen ab 30.03.2026 werden hier notiert, aber **nicht** als Notebooks umgesetzt.

| Idee | Beschreibung |
|------|--------------|
| KI-prädiktiver Dispatch | ML-basierte Preisvorhersage statt statischem p25/p75-Schwellenwert |

### Dev / Helper *(Einreichung ok, kein Pflicht-Output)*

| NB | Datei | Zweck |
|----|-------|-------|
| NB00b | `00b_Konventionen.ipynb` | Codestil, Pfad-Konventionen, config.json-Erklärung |
| NB01b | `01b_Daten_Sim.ipynb` | Simulierte Rohdaten für Entwicklungsphase |
| NB02b | `02b_Sim_Analyse.ipynb` | Simulierte Zwischendaten für Entwicklungsphase |
| NB99 | `99_Datenprovenienz.ipynb` | Pipeline-Lineage, dataindex.csv-Visualisierung |


---
## 6. Ordner- und Dateistruktur

```
Grid_Arbitrage_SC26_Gruppe_2/
│
├── config.json                    ← Statische Parameter (User pflegt, NBs nur lesen)
├── transfer.json                  ← Berechnete Ergebnisse (NBs schreiben/lesen)
├── run_all.sh                     ← Sequenzieller Ausführer NB01→NB04
├── dataindex.csv                  ← Datenquellen-Register (append-only)
│
├── 00_Project_Overview.ipynb      ← Pflicht: Übersicht + Projektplan
├── 00b_Konventionen.ipynb         ← Dev: Codestil, Konventionen
├── 01_Daten_Laden.ipynb           ← Pflicht
├── 01b_Daten_Sim.ipynb            ← Dev: Simulierte Rohdaten
├── 02_Daten_Analyse.ipynb         ← Pflicht
├── 02b_Sim_Analyse.ipynb          ← Dev: Simulierte Zwischendaten
├── 03_Visualisierungen.ipynb      ← Pflicht
├── 04_Business_Case.ipynb         ← Pflicht
├── 05_Business_Strategy.ipynb     ← Kür
├── 06_Raeumliche_Analyse.ipynb    ← Kür
├── 07_Cross_Border.ipynb          ← Kür (in Planung)
├── 08_Marktdynamik.ipynb          ← Kür (in Planung)
├── 09_Revenue_Stacking.ipynb      ← Kür (in Planung)
├── 10_Dispatch_Optimierung.ipynb  ← Kür (in Planung)
├── 11_Technologievergleich.ipynb  ← Kür (in Planung)
├── 12_Alternative_Speicher.ipynb  ← Kür (in Planung)
├── 13_Eigenverbrauch.ipynb        ← Kür (in Planung)
├── 14_Produkt_Steckbrief.ipynb    ← Kür (in Planung)
├── 15_Kombinierte_Simulation.ipynb← Kür (in Planung)
├── 99_Datenprovenienz.ipynb       ← Dev: manuell ausführen
│
├── data/
│   ├── raw/                       ← Rohdaten (szenario-unabhängig)
│   ├── processed/                 ← Bereinigte Zeitreihen (szenario-unabhängig)
│   └── intermediate/              ← Abgeleitete Dateien (szenario-abhängig)
│       ├── pessimistisch/
│       ├── realistisch/           ← aktives Szenario
│       └── optimistisch/
│
├── sim/                           ← Dev-only (identische Unterstruktur)
│   ├── raw/
│   ├── processed/
│   └── intermediate/
│
└── output/
    ├── charts/                    ← Pflicht Charts 1–8 (NB03)
    └── kuer/                      ← Charts NB05–NB10
        ├── pessimistisch/
        ├── realistisch/
        └── optimistisch/
```

**Wichtig:** `raw/` und `processed/` sind szenario-unabhängig: dieselben Spotpreise für alle Szenarien. Nur `intermediate/` enthält szenario-abhängige Berechnungen (z. B. Wirtschaftlichkeit, Netzentlastung).


---
## 7. Bewertung der Datenqualität

| Kriterium | Massnahme |
|-----------|----------|
| Plausibilität | Wertebereich-Prüfung (z. B. Preise -500 bis +3000 EUR/MWh) |
| Vollständigkeit | Stundenindex erzwingen, Lücken dokumentieren in `data/missing.txt` |
| Konsistenz | UTC-Normierung, Zeitzonensynchronisation aller Quellen |
| Fehlende Werte | Lineare Interpolation (max. 3h), Forward/Backward-Fill für längere Lücken |
| Ausreisser | Clipping auf physikalisch plausible Grenzen |
| **Währung** | Alle Berechnungen in EUR. CH-Tarife (CHF) werden mit `eur_chf` aus `config.json` umgerechnet. Kurs fix März 2026 (→ Designentscheidung Sektion 4.4). |
| API-Besonderheiten | `query_load` gibt DataFrame statt Series, MW→GW-Konvertierung |

Alle Download-Events sind in `dataindex.csv` protokolliert (status: `active` / `superseded` / `error` / `partial`).


---
## 8. Validierung mittels Simulation

In der initialen Projektphase wurden simulierte Daten (NB01b / NB02b) verwendet:

- Validierung der Datenpipeline vor erstem API-Zugang
- Überprüfung des erwarteten Verhaltens von Analysen und Visualisierungen
- Test der Robustheit der Dispatch-Logik

> **Wichtig:** Simulierte Daten wurden ausschliesslich zur Methodikvalidierung verwendet und sind **nicht** Bestandteil der finalen Ergebnisse. Alle Resultate basieren auf echten ENTSO-E- und BFE-Daten.

Die Sim-Notebooks wurden gegen den frühen Projektzustand erstellt und nicht laufend nachgeführt. Eine Synchronisierung ist nur nötig, wenn die Simulation aktiv wieder eingesetzt werden soll.


---
## 9. Einsatz von KI-Tools

KI-Tools wurden zur Beschleunigung der initialen Projektumsetzung eingesetzt, insbesondere zur Strukturierung der Notebooks sowie zur Erstellung eines ersten funktionsfähigen Prototyps. Zudem wurde KI zur Vorauswahl potenzieller Datenquellen genutzt.

**Eigenverantwortlich erarbeitet wurden:**
- Finale Auswahl und Bewertung aller Datenquellen
- Eigenständige Datenqualitätsprüfung
- Datenaufbereitung, Korrekturen und API-Anpassungen
- Alle methodischen Entscheidungen und Interpretationen

> Alle Teammitglieder können sämtliche Projektschritte nachvollziehbar erklären und begründen.


---
## 10. Projektplan

Der Projektplan wird als formatierbares Dict direkt in diesem Notebook gepflegt.  
Anpassungen: Werte im `PROJEKTPLAN`-Dict unten ändern, Zelle neu ausführen.

**Status-Schlüssel:** `✅ Erledigt` · `🔄 In Arbeit` · `⬜ Offen` · `⚠️ Verzögert`  
**Kürzel:** `PN` = Patrik Neunteufel · `SE` = Senthuran Elankeswaran · `CS` = Cyril Saladin · `Alle` = gesamtes Team


In [ ]:
# ── Projektplan ──────────────────────────────────────────────────────────────
import pandas as pd

STATUS_OFFEN = '⬜ Offen'
STATUS_IN_ARBEIT = '🔄 In Arbeit'
STATUS_VERZOEGERT = '⚠️ Verzögert'
STATUS_ERLEDIGT = '✅ Erledigt'

PROJEKTPLAN = [
    # (Phase, Aufgabe, Verantwortlich, Frist, Status)
    
    # ── Meilenstein 1: Antrag ─────────────────────────────────────────────────
    ('M1 – Antrag', 'Projekttitel & Datenquellen definieren',            'Alle', '30.03.2026', STATUS_ERLEDIGT),
    ('M1 – Antrag', 'Teambesprechung Aufgabenverteilung',                'Alle', '26.03.2026', STATUS_ERLEDIGT),
    ('M1 – Antrag', 'Moodle-Eintrag Datensätze ausfüllen',              'Alle', '30.03.2026', STATUS_ERLEDIGT),

    # ── Notebook-Skelette ─────────────────────────────────────────────────────
    ('Basis – NB0',  'NB00 Projektübersicht erstellen',                  '–',    '26.03.2026', STATUS_ERLEDIGT),
    ('Basis – NB0',  'NB00b Konventionen erstellen',                     '–',    '26.03.2026', STATUS_ERLEDIGT),
    ('Basis – NB1',  'NB01 Skelett + echte API-Anbindung',              '–',    '26.03.2026', STATUS_ERLEDIGT),
    ('Basis – NB2',  'NB02 Skelett + Dispatch + Wirtschaftlichkeit',    '–',    '26.03.2026', STATUS_ERLEDIGT),
    ('Basis – NB3',  'NB03 Skelett + Charts 1–8',                       '–',    '26.03.2026', STATUS_ERLEDIGT),
    ('Basis – NB4',  'NB04 Business Case Bericht',                      '–',    '26.03.2026', STATUS_IN_ARBEIT),
    ('Kür – NB5',   'NB05 Business Strategy Bericht',                   'PN',   '15.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB6',   'NB06 Räumliche Analyse',                           'PN',   '15.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB7',   'NB07 Cross-Border-Analyse',                        '–',    '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB8',   'NB08 Marktdynamik (Spread + CAPEX)',               '–',    '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB8a',  'NB08a Saisonale Animationen',                      'PN',   '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB9',   'NB09 Revenue Stacking (VPP, FCR)',                 '–',    '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB10',  'NB10 Dispatch-Optimierung (DA-optimal)',           '–',    '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB11',  'NB11 Technologievergleich (LFP/NMC/Redox/NaS)',   'PN',   '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB12',  'NB12 Alternative Speicher (PHS/CAES/P2X)',         'PN',   '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB13',  'NB13 Eigenverbrauchsoptimierung',                  'PN',   '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB14',  'NB14 Produktsteckbrief 7 kW / 14 kWh',            'PN',   '30.04.2026', STATUS_IN_ARBEIT),
    ('Kür – NB15',  'NB15_Kombinierte_Simulation',                     'PN',   '30.04.2026', STATUS_IN_ARBEIT),

    # ── Meilenstein Kür-Freeze ────────────────────────────────────────────────
    ('Kür – NB15',  'NB15 Kombinierte Simulation Hybrid-Dispatch',      'PN',   '30.04.2026', '🔄 In Arbeit'),

    ('M1.5 – Kür-Freeze', '🔒 Keine neuen Notebooks ab heute',          'Alle', '30.03.2026', STATUS_ERLEDIGT),
    ('M1.5 – Kür-Freeze', 'Einreichungsentscheid Kür-Notebooks',        'Alle', '30.04.2026', STATUS_OFFEN),

    # ── Meilenstein 2: Abgabe ─────────────────────────────────────────────────
    ('M2 – Abgabe', 'Pflicht-NBs vollständig ausführen (Restart & Run All)', 'Alle', '07.05.2026', STATUS_OFFEN),
    ('M2 – Abgabe', 'Kür-NBs bereinigen und testen',                    'Alle', '07.05.2026', STATUS_OFFEN),
    ('M2 – Abgabe', 'notebooks.zip erstellen (NB00–NB04)',              'Alle', '11.05.2026', STATUS_OFFEN),
    ('M2 – Abgabe', 'data.zip erstellen (data/ + output/)',             'Alle', '11.05.2026', STATUS_OFFEN),
    ('M2 – Abgabe', 'Moodle-Upload (Deadline 11.05.2026, 12:00)',       'Alle', '11.05.2026', STATUS_OFFEN),
]

df_plan = pd.DataFrame(PROJEKTPLAN, columns=['Phase','Aufgabe','Verantwortlich','Frist','Status'])

# Farbcodierung
STATUS_COLOR = {
    STATUS_ERLEDIGT:   'background-color: #1a4731; color: #5DCAA5',
    STATUS_IN_ARBEIT:  'background-color: #2c2800; color: #EF9F27',
    STATUS_OFFEN:      '',
    STATUS_VERZOEGERT: 'background-color: #3d1212; color: #F09595',
}
def highlight(row):
    c = STATUS_COLOR.get(row['Status'], '')
    return ['' if col != 'Status' else c for col in row.index]

df_plan.style.apply(highlight, axis=1).set_properties(**{'text-align': 'left'})

*Anpassung: `PROJEKTPLAN`-Liste in der Code-Zelle oben editieren und Zelle neu ausführen.*

---
## 11. Aufgabenverteilung

| Kürzel | Name | Notebook-Hauptverantwortung |
|--------|------|-----------------------------|
| **PN** | Patrik Neunteufel | NB01, NB05, NB06 + fachliche Leitung |
| **SE** | Senthuran Elankeswaran | NB02 |
| **CS** | Cyril Saladin | NB03 |
| **Alle** | Gesamtes Team | NB04, Review, Abgabe |

> Detailzuteilung erfolgt nach Teambesprechung vom 26.03.2026. Notebook-Zuordnung wird bei Bedarf oben im Projektplan aktualisiert.


---
## 12. Einreichung

**Moodle-Eintrag Datensätze** (Deadline: 30. März 2026, 12:00 Uhr):
- URL 1: `https://transparency.entsoe.eu` (DS1 Spot-Preise + DS2 Netzlast)
- URL 2: `https://data.geo.admin.ch/ch.bfe.elektrizitaetsproduktionsanlagen/` (DS3 BFE GeoPackage)
- ZHAW-Kürzel: `SC26_Gruppe_2`
- Projekttitel: *Grid-Arbitrage mit Batteriespeichern*

**Pflicht-Abgabe** (Deadline: 11. Mai 2026, 12:00 Uhr):
```
notebooks.zip   → 00_Project_Overview, 01_Daten_Laden,
                  02_Daten_Analyse, 03_Visualisierungen, 04_Business_Case,
                  config.json, transfer.json
data.zip        → data/ + output/charts/
```

**Kür-Abgabe** (Entscheid: 30. April 2026):
```
kuer.zip        → Auswahl aus NB05–NB14 gemäss Entscheid vom 30.04.2026
                  Kandidaten: 05, 06, 07, 08, 08a, 09, 10, 11, 12, 13, 14
```

**Dev / Helper** (optional, separat):
```
helper.zip      → 00b_Konventionen, 01b_Daten_Sim, 02b_Sim_Analyse, 99_Datenprovenienz
```

---
*CAS Information Engineering – Scripting, ZHAW School of Engineering, März–Mai 2026*


---
## 13. Potenzielle Erweiterungen

Ideen die über den Projektrahmen hinausgehen, aber dokumentiert sind:

| Dokument | Inhalt |
|---|---|
| `konzept_laendererweiterung.md` | Parametrisierung für DACH und ganz Europa (ENTSO-E, Eurostat, Bidding-Zone-Tabelle) |

Diese Dateien sind im Projektordner abgelegt und dienen als Ausgangspunkt für allfällige Weiterarbeit.
